### Lab 2.2: Perceptron Algorithm in PyTorch

In this lab you will again implement the perceptron algorithm, but this time using PyTorch.

In [1]:
import numpy as np
import torch

PyTorch is very similar to NumPy in its basic functionality.  In PyTorch arrays are called tensors.

In [2]:
a = torch.tensor(5)
a

tensor(5)

In [3]:
b = torch.tensor(6)
a+b

tensor(11)

In [4]:
c = torch.zeros(3,5).float()
c

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

*A note on broadcasting:* You may have noticed in the previous lab that NumPy is particular about the sizes of the arrays in operations; PyTorch is the same way.

For example, if `A` has shape `(10,5)` and `b` has shape `(10,)`, then we can't compute `A*b`.  It wants the *last* dimensions to match, not the first ones.  So you would need to do either `A.T*b`.

In [5]:
A = np.random.normal(size=(10,5))
b = np.ones(10)

In [6]:
try:
    A*b
except ValueError as e:
    print(e)

operands could not be broadcast together with shapes (10,5) (10,) 


In [7]:
A.T*b

array([[ 0.17370645, -1.2368405 , -0.44629957, -0.71497706,  0.25869929,
        -0.44849861,  0.6599924 ,  0.18316628, -2.1063781 , -1.71382169],
       [ 0.79685478,  0.64797787, -0.37369879, -0.3884908 ,  0.64328319,
        -1.17700587, -0.46260263, -0.65659506,  1.03251683,  0.01864147],
       [ 1.39195264, -0.81648306, -0.31043952,  0.89795172,  0.06970467,
         0.5324898 , -1.73887278, -0.80881408,  0.49357978,  0.38759943],
       [-1.67680055, -0.54284659, -0.93612707,  0.37463906,  0.30164838,
        -0.80909286, -2.63122785,  0.50723512, -0.69430358,  1.65378875],
       [ 0.20752725, -0.37089513, -1.66229952,  2.13486208,  0.97015974,
         1.98952581,  1.69951143,  0.26759897, -0.80194313,  0.9002343 ]])

An alternative is to introduce an extra dimension of size one to $b$.  However, note that this produces the transposed result from before.

In [8]:
A*b[:,None]

array([[ 0.17370645,  0.79685478,  1.39195264, -1.67680055,  0.20752725],
       [-1.2368405 ,  0.64797787, -0.81648306, -0.54284659, -0.37089513],
       [-0.44629957, -0.37369879, -0.31043952, -0.93612707, -1.66229952],
       [-0.71497706, -0.3884908 ,  0.89795172,  0.37463906,  2.13486208],
       [ 0.25869929,  0.64328319,  0.06970467,  0.30164838,  0.97015974],
       [-0.44849861, -1.17700587,  0.5324898 , -0.80909286,  1.98952581],
       [ 0.6599924 , -0.46260263, -1.73887278, -2.63122785,  1.69951143],
       [ 0.18316628, -0.65659506, -0.80881408,  0.50723512,  0.26759897],
       [-2.1063781 ,  1.03251683,  0.49357978, -0.69430358, -0.80194313],
       [-1.71382169,  0.01864147,  0.38759943,  1.65378875,  0.9002343 ]])

In [9]:
A*np.expand_dims(b,-1)

array([[ 0.17370645,  0.79685478,  1.39195264, -1.67680055,  0.20752725],
       [-1.2368405 ,  0.64797787, -0.81648306, -0.54284659, -0.37089513],
       [-0.44629957, -0.37369879, -0.31043952, -0.93612707, -1.66229952],
       [-0.71497706, -0.3884908 ,  0.89795172,  0.37463906,  2.13486208],
       [ 0.25869929,  0.64328319,  0.06970467,  0.30164838,  0.97015974],
       [-0.44849861, -1.17700587,  0.5324898 , -0.80909286,  1.98952581],
       [ 0.6599924 , -0.46260263, -1.73887278, -2.63122785,  1.69951143],
       [ 0.18316628, -0.65659506, -0.80881408,  0.50723512,  0.26759897],
       [-2.1063781 ,  1.03251683,  0.49357978, -0.69430358, -0.80194313],
       [-1.71382169,  0.01864147,  0.38759943,  1.65378875,  0.9002343 ]])

In general, carefully check the sizes of all arrays in your code!

In [10]:
from palmerpenguins import load_penguins
from mlxtend.plotting import plot_decision_regions
from matplotlib import pyplot as plt

Here we loading and format the Palmer penguins dataset for binary classification.

In [11]:
df = load_penguins()

# drop rows with missing values
df.dropna(inplace=True)

# tricky code to randomly shuffle the rows
df = df.sample(frac=1).reset_index(drop=True)

# select only two specices
df = df[(df['species']=='Adelie')|(df['species']=='Chinstrap')]

# get two features
X = df[['flipper_length_mm','bill_length_mm']].values

# convert speces labels to 0 and 1
y = df['species'].map({'Adelie':0,'Chinstrap':1}).values

To make the learning algorithm work more smoothly, we we will subtract the mean of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [12]:
X -= np.mean(X,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [13]:
X = torch.tensor(X).float()
y = torch.tensor(y).float()

In [14]:
X

tensor([[-6.9206e+00, -4.4047e+00],
        [ 3.0794e+00, -7.0467e-01],
        [-4.9206e+00,  4.1953e+00],
        [-1.3921e+01,  4.0953e+00],
        [-9.2056e-01, -3.0047e+00],
        [-2.9206e+00, -5.1047e+00],
        [-9.2056e-01, -3.4047e+00],
        [-1.9206e+00, -4.2047e+00],
        [ 3.0794e+00, -6.0047e+00],
        [-4.9206e+00, -1.5047e+00],
        [-1.0921e+01, -4.4047e+00],
        [ 7.9439e-02, -4.7047e+00],
        [-1.9206e+00, -7.0047e+00],
        [-9.2056e-01, -2.4047e+00],
        [ 4.0794e+00,  6.9533e-01],
        [ 5.0794e+00,  9.2953e+00],
        [-2.9206e+00, -3.7047e+00],
        [ 1.3079e+01, -9.0467e-01],
        [ 6.0794e+00, -7.4047e+00],
        [ 1.0794e+00, -5.3047e+00],
        [-9.2056e-01, -6.4047e+00],
        [-2.9206e+00, -6.1047e+00],
        [ 2.0794e+00,  9.6953e+00],
        [ 5.0794e+00,  8.2953e+00],
        [-9.2056e-01,  3.1953e+00],
        [-1.9206e+00, -2.4047e+00],
        [-7.9206e+00, -5.6047e+00],
        [-1.9206e+00, -3.904

### Exercises

Your task is to again complete this class for the perceptron, with two changes from last time:
- the implementation should use PyTorch tensors, not NumPy arrays;
- `train_step` now accepts the entire dataset as input and should calculate the average gradient over all examples, rather than updating the weights one data point at a time.

In [119]:
class Perceptron:
    def __init__(self,lr=1e-3):
        # store the learning rate
        self.lr = lr

        # initialize the weights to small, normally-distributed values
        self.w = torch.normal(mean=0, std=0.01, size=(2,))

        # initialize the bias to zero
        self.b = torch.zeros(1)

    def train_step(self,X:torch.Tensor,y:torch.Tensor) -> None:
        """ Apply the first update rule shown in lecture.
            Arguments:
             X: data matrix of shape (N,2)
             y: labels of shape (N,) 
        """
        # WRITE CODE HERE
        y = y * 2 -1 # convert Adelie -1, Chinstrap 1
        # gradient_w = -(X * y.unsqueeze(1)).mean(dim=0) 
        # gradient_b = -y.float().mean()
        # self.w = self.w - self.lr * gradient_w   # w' = w - s*dl/dw 
        # self.b = self.b - self.lr * gradient_b
        z = X @ self.w + self.b
        err = y-z
        self.w += self.lr * torch.mean(err[:,None]*X, dim=0)
        self.b += self.lr * torch.mean(err)
        
    
    def predict(self,X:torch.Tensor) -> torch.Tensor:
        """ Calculate model prediction for all data points.
            Arguments:
             X: data matrix of shape (N,2)   
            Returns:
             Predicted labels (0 or 1) of shape (N,)
        """
        # WRITE CODE HERE
        Z = X @ self.w + self.b
        return torch.where(Z < 0, 0, 1)
    
    def score(self,X:torch.Tensor,y:torch.Tensor) -> torch.Tensor:
        """ Calculate model accuracy
            Arguments:
             X: data matrix of shape (N,2)   
             y: labels of shape (N,)
            Returns:
             Accuracy score
        """
        # WRITE CODE HERE
        Z = self.predict(X)
        return (Z == y).float().mean()


Run the following code to train the model and print out the accuracy at each step.

In [148]:
lr = 1e-3
# epochs = 100
max_score = -1
for epochs in range (20, 1000, 20):
    model = Perceptron(lr)
    for i in range(epochs):
        model.train_step(X,y)

    epoch_score = model.score(X, y)
    if epoch_score > max_score:
        print(f"Score: {epoch_score} in epoch {epochs}")
        max_score= epoch_score

print("")
rates = [1e-4, 15e-5, 1e-3, 5e-3, 0.01, 0.015, 0.1]
max_score = -1
for lr in rates:
    model = Perceptron(lr)
    for i in range(100):
        model.train_step(X, y)
    
    epoch_score = model.score(X, y)
    if epoch_score > max_score:
        print(f"Score: {epoch_score} with learning rate {lr}")
        max_score= epoch_score

Score: 0.836448609828949 in epoch 20

Score: 0.8130841255187988 with learning rate 0.0001
Score: 0.827102780342102 with learning rate 0.00015
Score: 0.836448609828949 with learning rate 0.001


Run the training multiple times.  Is the training the same each time, or does it vary?  Why?

It varies because we set the weights randomly so the gradients must adjust to the minimum. 

Play with the learning rate and number of epochs to find the best setting.

In testing different epoch numbers I found that the model usually hit a maximum accuracy score of 83.64% at around epoch 40 (with learning rate of 0.001). In trying different learning rates (with epoch = 100), the highest accuracy scores resulted from a learning rate of 0.001. 
